# NB08 — Stock Valuation with ML
**Machine Learning in Finance · Week 8**  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruiyuan-chen/ml-finance-course/blob/main/notebooks/NB08_ml_valuation.ipynb)

**Finance question:** Is this stock cheap or expensive compared with what it "should" trade at?

**Key idea:** **Comparables, done by machine.** Predict each company's valuation multiple from its fundamentals, using the other companies as comparables. The gap between the actual and the predicted multiple is a possible **mispricing** signal.

In [ ]:
# ▶ RUN ME FIRST — loads the course tools and data (about 30 seconds the first time)
COURSE_URL = "https://raw.githubusercontent.com/ruiyuan-chen/ml-finance-course/main/"
import os, sys, urllib.request
here = [f for f in [".", "..", "../.."] if os.path.exists(os.path.join(f, "mlfin.py"))]
if here:
    sys.path.insert(0, here[0])
else:
    urllib.request.urlretrieve(COURSE_URL + "mlfin.py", "mlfin.py")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mlfin
mlfin.setup(COURSE_URL)

## Step 1: a valuation snapshot every year-end
`make_valuation_panel` takes each December and records what an investor knew: price-to-book (P/B),
EV/EBITDA, return on equity (ROE), growth, margins, debt and more, plus what the stock did over the
**next** 12 months (for testing only).

In [ ]:
vp = mlfin.make_valuation_panel()
print(f"{len(vp):,} company-years, {vp['year'].min()}–{vp['year'].max()}")
vp[(vp["year"] == 2023) & vp["ticker"].isin(["AAPL", "JPM", "XOM", "NVDA", "KO"])][
    ["ticker", "name", "sector", "pb", "ev_ebitda", "roe", "growth", "margin"]].round(2)

## Step 2: classic comps
The textbook approach: compare each firm with the **median of its sector**. A bank at 1.2× book in a
sector whose median is 1.5× looks "cheap". Simple, but it ignores everything except the sector.

In [ ]:
# 🟡 YOUR TURN
YEAR = 2023   # ← any year 2010–2024

snap = vp[vp["year"] == YEAR].dropna(subset=["pb"])
comps = snap.groupby("sector")["pb"].median().round(2).sort_values()
comps.to_frame("median P/B, " + str(YEAR))

## Step 3: ML comps
Instead of "same sector", let a model learn what **justifies** a high multiple: profitability (ROE), growth,
margins, debt, size, risk. Each year the model learns from the *other* firms (cross-validation), so no
firm is its own comparable.

In [ ]:
# 🟡 YOUR TURN
MULTIPLE = "pb"   # ← "pb" (price-to-book) or "ev_ebitda" (enterprise value / EBITDA; banks excluded)
PREDICTORS = ["roe", "growth", "margin", "leverage", "size", "beta_12", "vol_12", "mom_12_1"]
# ← drop some to see which matter; mlfin.FEATURE_DESCRIPTIONS explains most of them

vals = mlfin.fit_valuation(vp, MULTIPLE, PREDICTORS)

*What to look for:* the share of the differences in multiples each method explains (R-squared). The ML
version uses fundamentals **and** the sector; the classic version uses only the sector median.

## Step 4: cheap and expensive, by name

In [ ]:
# 🟡 YOUR TURN
SECTOR = None   # ← None for all, or e.g. "Energy", "Information Technology", "Health Care"

mlfin.plot_valuation(vals, YEAR)
show = vals[vals["year"] == YEAR]
if SECTOR:
    show = show[show["sector"] == SECTOR]
cols = ["ticker", "name", "sector", MULTIPLE, "predicted", "gap"]
print("CHEAPEST (actual multiple far below predicted):")
display(show.nsmallest(10, "gap")[cols].round(2))
print("MOST EXPENSIVE:")
display(show.nlargest(10, "gap")[cols].round(2))

*Before you call anything a bargain:* what might the model **not know**? A lawsuit, a one-off loss that
crushed book value, an industry in decline, an acquisition? The model sees eight numbers; you see the news.

## Step 5: what makes a stock deserve a high multiple?
Permutation importance (NB06) on one year's valuation model: which fundamentals move the predicted
multiple most?

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
yr = vals[vals["year"] == YEAR]
X = yr[PREDICTORS]
y = np.log(yr[MULTIPLE])
one_year = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_depth=3, random_state=0).fit(X, y)
_ = mlfin.plot_importance(one_year, X=X, y=y, title=f"What drives {MULTIPLE} in {YEAR}?")

## Step 6: did "cheap" beat "expensive"?
Each year, sort firms into five groups by the gap (group 1 = cheapest), then look at what they returned
over the **next** 12 months. Then the same test for classic sector comps.

In [ ]:
groups, by_year = mlfin.valuation_returns(vals, "gap")
groups.T

In [ ]:
groups_c, by_year_c = mlfin.valuation_returns(vals, "comps_gap")
groups_c.T

*What to look for:* if the gap were a reliable mispricing signal, group 1 would beat group 5 most
years. Check the right-hand chart: how often did cheap win, and which years were disasters?

## ⭐ GOING FURTHER (optional)
1. Switch `MULTIPLE` to `"ev_ebitda"`. Does the model explain EV/EBITDA as well as P/B? Why might that be?
2. Set `SECTOR = "Information Technology"` and call `mlfin.fit_valuation(vp, MULTIPLE, PREDICTORS,
   sector=SECTOR)`: a model trained only on one sector's firms. Are the cheap names different?
3. Ask an AI assistant: *"What is a value trap?"* Find one name in your cheap list that might be one.

## 🪞 REFLECTION (graded — write in your own words, no AI)

Double-click this cell and replace the *...* with your answers.

1. **In one sentence, what finance question did this week's technique answer?**
   *...*
2. **What is one way the result could be misleading?**
   *...*
3. **What did you ask an AI assistant this week, and was its answer right?**
   *...*